# Regional & Income Group Comparative Analysis
## Global Healthcare Performance & Equity Analysis — Mahmoud Task

This notebook compares country-level healthcare and socioeconomic indicators across the `Region` and `Income_Group` classifications in `country_dictionary_v1.csv`.

**Approach:** for each indicator, select the latest non-missing observation available for each country. This produces at most one record per country and indicator, avoiding the false independence that would arise from treating all repeated country-year records as separate observations in the cross-sectional tests. The reference year is retained and reported because countries do not all have data for the same year.

The supplied dictionary's group classifications are treated as fixed labels for this analysis. They are not reconstructed historically. Countries without a Region or Income_Group in the dictionary are excluded only from the relevant classified-group comparison; their source data remains untouched.


## Methodology and interpretation safeguards

1. **Join:** left-join the dictionary on `ISO3`; the integrated dataset remains unchanged.
2. **Latest-available country snapshot:** one latest non-null value per country per indicator. This means different indicators can have different reference years and different country counts.
3. **Descriptive comparison:** mean, median, standard deviation, quartiles, minimum, maximum, country count, and reference-year range by region and income group.
4. **Overall group test:** Kruskal–Wallis test, suitable for comparing independent groups without assuming normally distributed values. The reported epsilon-squared is an effect-size estimate.
5. **Pairwise follow-up:** two-sided Mann–Whitney U tests with Holm adjustment within each indicator/grouping-dimension family.
6. **Time trends:** annual group medians from 2000–2025 are descriptive only and require at least three country observations per group-year.

**Important:** these are observational comparisons, not causal estimates. Group membership is taken exactly from the supplied static dictionary. Differences in reference years, data availability, country heterogeneity, and sparse indicators can affect interpretation. Financial hardship, equity, and literacy indicators have notably thinner coverage than the core healthcare indicators.


In [ ]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import stats
from statsmodels.stats.multitest import multipletests

# Resolve the project root and locate the two supplied inputs.
cwd = Path.cwd()
search_roots = [cwd, *cwd.parents]
project_root = next(
    (p for p in search_roots if (p / "03_Integrated_Datasets").exists()
     or (p / "04_Data_Analysis").exists()),
    cwd
)

def locate_input(filename):
    candidates = []
    for root in search_roots:
        candidates.extend([
            root / filename,
            root / "03_Integrated_Datasets" / filename,
            root / "02_Data_Preparation" / filename,
        ])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {filename}. Place it in the project root, "
        "03_Integrated_Datasets/, or 02_Data_Preparation/."
    )

integrated_path = locate_input("integrated_healthcare_dataset.csv")
dictionary_path = locate_input("country_dictionary_v1.csv")

# Save outputs inside this task folder when running from the repo root;
# if the notebook kernel already starts in this folder, use the current folder.
task_folder_name = "06_Regional_Income_Analysis_Mahmoud"
if cwd.name == task_folder_name:
    output_dir = cwd
elif (project_root / "04_Data_Analysis").exists():
    output_dir = project_root / "04_Data_Analysis" / task_folder_name
else:
    output_dir = cwd / task_folder_name
charts_dir = output_dir / "charts"
charts_dir.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(integrated_path)
dictionary = pd.read_csv(dictionary_path)

print("Integrated dataset:", data.shape)
print("Country dictionary:", dictionary.shape)
print("Input files:", integrated_path, dictionary_path)


In [ ]:
# Validate the keys and dictionary before merging.
assert {"ISO3", "Country", "Year"}.issubset(data.columns)
assert {"ISO3", "Region", "Income_Group"}.issubset(dictionary.columns)
assert not data.duplicated(["ISO3", "Year"]).any(), "Duplicate ISO3-Year keys found."
assert dictionary["ISO3"].is_unique, "Dictionary ISO3 values must be unique."

merged = data.merge(
    dictionary[["ISO3", "Region", "Income_Group"]],
    on="ISO3",
    how="left",
    validate="many_to_one"
)

validation = pd.DataFrame([
    ["Integrated dataset rows", len(data), "Source row count"],
    ["Duplicate ISO3-Year keys", int(data.duplicated(["ISO3", "Year"]).sum()), "Must be zero"],
    ["Countries in integrated dataset", data["ISO3"].nunique(), "Unique ISO3 count"],
    ["Countries in dictionary", dictionary["ISO3"].nunique(), "Unique ISO3 count"],
    ["ISO3 codes unmatched to dictionary", len(set(data["ISO3"]) - set(dictionary["ISO3"])), "Must be zero"],
    ["Dictionary rows missing Region", int(dictionary["Region"].isna().sum()), "Not imputed"],
    ["Dictionary rows missing Income_Group", int(dictionary["Income_Group"].isna().sum()), "Not imputed"],
    ["Rows after left join", len(merged), "Must equal source rows"],
    ["Rows missing Region after join", int(merged["Region"].isna().sum()), "Excluded from region-only tests"],
    ["Rows missing Income_Group after join", int(merged["Income_Group"].isna().sum()), "Excluded from income-only tests"],
], columns=["Validation_Check", "Result", "Interpretation"])
validation


In [ ]:
# Indicators selected for socioeconomic and healthcare-capacity comparisons.
indicators = [
    "GDP_Per_Capita_PPP_2021",
    "Doctors_Per_10k",
    "Nurses_Midwives_Per_10k",
    "Hospital_Beds_Per_10k",
    "UHC_Service_Coverage_Index",
    "Health_Exp_Per_Capita_PPP",
    "Health_Exp_Pct_GDP",
    "OOP_Pct_Current_Health_Exp",
    "Financial_Hardship_Total_Pct",
    "Equity_Gap_Pct_Points",
    "Urban_Population_Pct",
    "Secondary_Enrollment_Gross_Pct",
    "Adult_Literacy_Rate",
]

# One latest available non-null observation per country and indicator.
snapshot_parts = []
for indicator in indicators:
    subset = merged[
        ["ISO3", "Country", "Year", "Region", "Income_Group", indicator]
    ].dropna(subset=[indicator]).sort_values(["ISO3", "Year"])
    latest = subset.groupby("ISO3", as_index=False).tail(1).copy()
    latest["Indicator"] = indicator
    latest = latest.rename(columns={indicator: "Value", "Year": "Reference_Year"})
    snapshot_parts.append(latest)

snapshot = pd.concat(snapshot_parts, ignore_index=True)
coverage_summary = (
    snapshot.groupby("Indicator")
    .agg(
        Countries_with_latest_value=("ISO3", "nunique"),
        Reference_Year_Median=("Reference_Year", "median"),
        Reference_Year_Min=("Reference_Year", "min"),
        Reference_Year_Max=("Reference_Year", "max"),
    )
    .reset_index()
)
coverage_summary["Countries_with_value_pct_of_196"] = (
    coverage_summary["Countries_with_latest_value"] / data["ISO3"].nunique() * 100
)
coverage_summary["Region_Classified_Countries"] = snapshot.groupby("Indicator").apply(
    lambda group: group.dropna(subset=["Region"])["ISO3"].nunique(),
    include_groups=False
).values
coverage_summary["Income_Classified_Countries"] = snapshot.groupby("Indicator").apply(
    lambda group: group.dropna(subset=["Income_Group"])["ISO3"].nunique(),
    include_groups=False
).values
coverage_summary


In [ ]:
def summarize_groups(frame, group_col):
    records = []
    for (indicator, group), group_frame in frame.dropna(subset=[group_col]).groupby(
        ["Indicator", group_col]
    ):
        values = group_frame["Value"].dropna()
        records.append({
            "Indicator": indicator,
            "Group_Type": group_col,
            "Group": group,
            "Countries_n": values.count(),
            "Mean": values.mean(),
            "Median": values.median(),
            "Std_Dev": values.std(),
            "P25": values.quantile(0.25),
            "P75": values.quantile(0.75),
            "Minimum": values.min(),
            "Maximum": values.max(),
            "Median_Reference_Year": group_frame["Reference_Year"].median(),
            "Earliest_Reference_Year": group_frame["Reference_Year"].min(),
            "Latest_Reference_Year": group_frame["Reference_Year"].max(),
        })
    return pd.DataFrame(records)

regional_comparison = summarize_groups(snapshot, "Region")
income_group_comparison = summarize_groups(snapshot, "Income_Group")
group_comparison_summary = pd.concat(
    [regional_comparison, income_group_comparison], ignore_index=True
)

display(regional_comparison.head())
display(income_group_comparison.head())


In [ ]:
def kruskal_tests(frame, group_col):
    records = []
    for indicator, indicator_frame in frame.dropna(subset=[group_col]).groupby("Indicator"):
        arrays, labels = [], []
        for label, group_frame in indicator_frame.groupby(group_col):
            values = group_frame["Value"].dropna().to_numpy()
            if len(values) >= 3:
                arrays.append(values)
                labels.append(label)
        if len(arrays) >= 2:
            h_stat, p_value = stats.kruskal(*arrays)
            n, k = sum(map(len, arrays)), len(arrays)
            epsilon_sq = max(0, (h_stat - k + 1) / (n - k)) if n > k else np.nan
            records.append({
                "Indicator": indicator,
                "Group_Type": group_col,
                "Groups_Compared": k,
                "Countries_n": n,
                "Kruskal_Wallis_H": h_stat,
                "P_Value": p_value,
                "Epsilon_Squared": epsilon_sq,
            })
    result = pd.DataFrame(records)
    if not result.empty:
        result["Holm_Adjusted_P_Value"] = multipletests(
            result["P_Value"], method="holm"
        )[1]
    return result

def pairwise_mannwhitney(frame, group_col):
    records = []
    for indicator, indicator_frame in frame.dropna(subset=[group_col]).groupby("Indicator"):
        groups = {
            str(label): g["Value"].dropna().to_numpy()
            for label, g in indicator_frame.groupby(group_col)
        }
        groups = {label: values for label, values in groups.items() if len(values) >= 3}
        pairs, p_values = [], []
        for group_1, group_2 in combinations(sorted(groups), 2):
            result = stats.mannwhitneyu(
                groups[group_1], groups[group_2],
                alternative="two-sided", method="auto"
            )
            pairs.append((group_1, group_2, result.statistic))
            p_values.append(result.pvalue)
        if p_values:
            adjusted = multipletests(p_values, method="holm")[1]
            for (group_1, group_2, u_stat), p_value, adjusted_p in zip(
                pairs, p_values, adjusted
            ):
                n1, n2 = len(groups[group_1]), len(groups[group_2])
                rank_biserial = 1 - (2 * u_stat) / (n1 * n2)
                records.append({
                    "Indicator": indicator,
                    "Group_Type": group_col,
                    "Group_1": group_1,
                    "Group_2": group_2,
                    "N_1": n1,
                    "N_2": n2,
                    "U_Statistic": u_stat,
                    "P_Value": p_value,
                    "Holm_Adjusted_P_Value": adjusted_p,
                    "Rank_Biserial_Effect": rank_biserial,
                })
    return pd.DataFrame(records)

regional_tests = kruskal_tests(snapshot, "Region")
income_tests = kruskal_tests(snapshot, "Income_Group")
statistical_tests = pd.concat([regional_tests, income_tests], ignore_index=True)
posthoc_tests = pd.concat([
    pairwise_mannwhitney(snapshot, "Region"),
    pairwise_mannwhitney(snapshot, "Income_Group")
], ignore_index=True)

display(statistical_tests.sort_values("Holm_Adjusted_P_Value").head(15))


In [ ]:
# Descriptive annual medians (2000–2025), requiring >=3 countries per group-year.
trend_records = []
for group_col in ["Region", "Income_Group"]:
    for indicator in indicators:
        subset = merged[["ISO3", "Year", group_col, indicator]].dropna()
        subset = subset[subset["Year"].between(2000, 2025)]
        for (group, year), group_frame in subset.groupby([group_col, "Year"]):
            country_count = group_frame["ISO3"].nunique()
            if country_count >= 3:
                trend_records.append({
                    "Group_Type": group_col,
                    "Group": group,
                    "Indicator": indicator,
                    "Year": int(year),
                    "Countries_n": country_count,
                    "Median": group_frame[indicator].median(),
                })
group_trends = pd.DataFrame(trend_records)
group_trends.head()


In [ ]:
# Export all analysis tables.
output_dir.mkdir(parents=True, exist_ok=True)
snapshot.to_csv(output_dir / "country_indicator_latest_snapshot.csv", index=False)
coverage_summary.to_csv(output_dir / "indicator_coverage_summary.csv", index=False)
group_comparison_summary.to_csv(output_dir / "group_comparison_summary.csv", index=False)
regional_comparison.to_csv(output_dir / "regional_comparison.csv", index=False)
income_group_comparison.to_csv(output_dir / "income_group_comparison.csv", index=False)
statistical_tests.to_csv(output_dir / "statistical_tests.csv", index=False)
posthoc_tests.to_csv(output_dir / "posthoc_pairwise_tests.csv", index=False)
group_trends.to_csv(output_dir / "group_trends_2000_2025.csv", index=False)
validation.to_csv(output_dir / "validation_summary.csv", index=False)

# Compact lowest-versus-highest median summary per indicator and grouping dimension.
summary_rows = []
for group_type, summary_frame, tests_frame in [
    ("Region", regional_comparison, regional_tests),
    ("Income_Group", income_group_comparison, income_tests),
]:
    for indicator in indicators:
        group_rows = summary_frame[summary_frame["Indicator"] == indicator].sort_values("Median")
        test_row = tests_frame[tests_frame["Indicator"] == indicator]
        if group_rows.empty:
            continue
        test = test_row.iloc[0] if not test_row.empty else None
        summary_rows.append({
            "Group_Type": group_type,
            "Indicator": indicator,
            "Lowest_Median_Group": group_rows.iloc[0]["Group"],
            "Lowest_Group_Median": group_rows.iloc[0]["Median"],
            "Lowest_Group_n": group_rows.iloc[0]["Countries_n"],
            "Highest_Median_Group": group_rows.iloc[-1]["Group"],
            "Highest_Group_Median": group_rows.iloc[-1]["Median"],
            "Highest_Group_n": group_rows.iloc[-1]["Countries_n"],
            "Median_Range_High_minus_Low": (
                group_rows.iloc[-1]["Median"] - group_rows.iloc[0]["Median"]
            ),
            "Kruskal_Wallis_P_Value": test["P_Value"] if test is not None else np.nan,
            "Kruskal_Wallis_Holm_Adjusted_P_Value": (
                test["Holm_Adjusted_P_Value"] if test is not None else np.nan
            ),
            "Epsilon_Squared": test["Epsilon_Squared"] if test is not None else np.nan,
        })
regional_income_analysis_summary = pd.DataFrame(summary_rows)
regional_income_analysis_summary.to_csv(
    output_dir / "regional_income_analysis_summary.csv", index=False
)

print("Saved analysis outputs to:", output_dir.resolve())


In [ ]:
# Charts for four core indicators; one chart per grouping dimension.
def plot_group_medians(summary_frame, indicator, title, ylabel, filename):
    subset = summary_frame[summary_frame["Indicator"] == indicator].sort_values("Median")
    if subset.empty:
        return
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(subset["Group"].astype(str), subset["Median"])
    ax.set_title(title, fontsize=12, weight="bold")
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Group")
    ax.grid(axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=20)
    for index, row in enumerate(subset.itertuples()):
        ax.text(index, row.Median, f"n={row.Countries_n}", ha="center", va="bottom", fontsize=8)
    fig.tight_layout()
    fig.savefig(charts_dir / filename, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)

charts_dir.mkdir(parents=True, exist_ok=True)
for indicator, label, stem in [
    ("UHC_Service_Coverage_Index", "UHC service coverage index", "uhc"),
    ("Doctors_Per_10k", "Doctors per 10,000 population", "doctors"),
    ("Hospital_Beds_Per_10k", "Hospital beds per 10,000 population", "beds"),
    ("OOP_Pct_Current_Health_Exp", "Out-of-pocket share of current health expenditure (%)", "oop"),
]:
    plot_group_medians(
        regional_comparison, indicator,
        f"Latest available {label} by region", label, f"{stem}_by_region.png"
    )
    plot_group_medians(
        income_group_comparison, indicator,
        f"Latest available {label} by income group", label, f"{stem}_by_income_group.png"
    )

# Time-series plots are descriptive; group categories are fixed from the supplied dictionary.
for indicator, ylabel, group_col, filename in [
    ("UHC_Service_Coverage_Index", "UHC service coverage index", "Region", "uhc_trends_region.png"),
    ("Doctors_Per_10k", "Doctors per 10,000 population", "Region", "doctors_trends_region.png"),
    ("Hospital_Beds_Per_10k", "Hospital beds per 10,000 population", "Region", "beds_trends_region.png"),
    ("OOP_Pct_Current_Health_Exp", "Out-of-pocket share of current health expenditure (%)", "Income_Group", "oop_trends_income_group.png"),
]:
    subset = group_trends[
        (group_trends["Indicator"] == indicator) &
        (group_trends["Group_Type"] == group_col)
    ]
    if subset.empty:
        continue
    fig, ax = plt.subplots(figsize=(10, 5))
    for group, group_frame in subset.groupby("Group"):
        ax.plot(
            group_frame["Year"], group_frame["Median"],
            marker="o", markersize=2.5, linewidth=1.5, label=str(group)
        )
    ax.set_title(f"{ylabel}: annual group medians (2000–2025)", fontsize=12, weight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    ax.legend(title=group_col.replace("_", " "), fontsize=8)
    fig.tight_layout()
    fig.savefig(charts_dir / filename, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)


## Main interpretation notes

- Compare medians together with `Countries_n` and `Median_Reference_Year`; do not read the median alone.
- `Holm_Adjusted_P_Value` in `statistical_tests.csv` adjusts the omnibus tests across indicators separately for Region and Income_Group.
- Pairwise adjusted p-values are corrected within each indicator and grouping dimension.
- Statistical significance does not imply a causal relationship or a practically important difference.
- The income-group labels are the static labels in the supplied dictionary; this notebook does not assign historical World Bank income categories by year.
